# 03 · Treino e competição entre modelos

**Objetivo.** Treinar dois algoritmos diferentes sob as mesmas condições e escolher o vencedor do ciclo pela métrica primária.

**Entradas.** O ciclo corrente, o dataset processado e `data/processed/features_eleitas.json`.

**Saídas.**
- MLflow: um run filho por modelo, com hiperparâmetros, semente, atributos, métricas de validação e o artefato `modelo` (pipeline sklearn com pré-processamento).
- `data/processed/competicao.json` e `reports/evidencias/<cycle_id>/03_competicao.json`: resultado da competição, lido pelo notebook 04.

**Protocolo (idêntico para A e B).**

| Item | Valor |
|---|---|
| Atributos | Exatamente os eleitos no notebook 02 |
| Ajuste | Partição **treino** (pré-processador + estimador no mesmo `Pipeline`) |
| Comparação | Partição **validação** |
| Métrica primária | `evaluation.primary_metric` (F1), direção `evaluation.direction` |
| Secundárias | Precisão, recall, acurácia, ROC-AUC e matriz de confusão |
| Semente | `data.random_state` |

O conjunto de **teste não é usado** aqui.

**Modelos.**
- **A · `LogisticRegression`**: linear e interpretável, com one-hot nas categóricas e padronização nas numéricas.
- **B · `RandomForestClassifier`**: conjunto de árvores, captura não linearidades. A profundidade é limitada para controlar overfitting e o tamanho do artefato.

Os dois usam `class_weight="balanced"`, que compensa os cerca de 22% de positivos. Sem isso, o limiar de 0,5 favorece a classe majoritária e derruba o recall. Os hiperparâmetros ficam em `config.models`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

import json

from src import features as fs, modelagem as mdl, preprocessamento as pp
from src.utils import load_config, load_cycle, read_json

pd.set_option("display.max_columns", 50)
cfg = load_config(ROOT)
TARGET = cfg["data"]["target"]

In [2]:
ctx = load_cycle(cfg, ROOT)
dataset = pp.load_processed(cfg, ctx, ROOT)
elected = fs.load_elected_features(cfg, ctx, ROOT)
voting = read_json(fs.elected_features_path(cfg, ROOT))

train = pp.split_xy(dataset, "treino", elected, cfg)
valid = pp.split_xy(dataset, "validacao", elected, cfg)
print("cycle_id      :", ctx.cycle_id)
print("data_version  :", ctx.data_version)
print("atributos     :", elected)
print("treino/valid. :", len(train[0]), "/", len(valid[0]))

cycle_id      : ciclo_gha_43986932580_1
data_version  : dv_3a043c533b992faa
atributos     : ['LIMIT_BAL', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_6', 'PAY_AMT1', 'UTILIZACAO_MEDIA', 'MESES_EM_ATRASO', 'ATRASO_MAXIMO']
treino/valid. : 16239 / 5439


## 1. Treino dos modelos A e B

In [3]:
results = []
for algorithm, spec in mdl.enabled_models(cfg).items():
    result = mdl.train_and_evaluate(algorithm, spec, elected, train, valid, cfg)
    mdl.log_model_run(result, ctx, cfg, voting, train[0], ROOT)
    results.append(result)
    print(f"{result.label} ({algorithm}) treinado em {result.train_seconds:.1f}s, run {result.run_id}")

assert len({tuple(r.features) for r in results}) == 1 and results[0].features == elected, "mesmos atributos nos dois modelos"
assert len({(r.n_train, r.n_validation, r.random_state) for r in results}) == 1, "mesmo protocolo nos dois modelos"

/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


modelo_A (logistic_regression) treinado em 0.1s, run 304b8409d5ce47bf83e16b23c97527f4


/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


modelo_B (random_forest) treinado em 0.6s, run c8ebc5d040024e4d9f2c796572fc2301


## 2. Comparação na validação

In [4]:
metric = cfg["evaluation"]["primary_metric"]
comparison = pd.DataFrame([{"modelo": r.label, "algoritmo": r.algorithm, **r.validation} for r in results]).set_index("modelo")
display(comparison)

for r in results:
    print(r.label, "hiperparâmetros:", json.dumps(r.params), "| random_state:", r.random_state)

,algoritmo,f1,precision,recall,accuracy,roc_auc,tn,fp,fn,tp
modelo,,,,,,,,,,
modelo_A,logistic_regression,0.524625,0.471734,0.590873,0.749770,0.746626,3327,841,520,751
modelo_B,random_forest,0.534276,0.484926,0.594807,0.757676,0.773764,3365,803,515,756


modelo_A hiperparâmetros: {"C": 1.0, "class_weight": "balanced", "max_iter": 2000} | random_state: 42
modelo_B hiperparâmetros: {"n_estimators": 150, "max_depth": 8, "min_samples_leaf": 20, "class_weight": "balanced", "n_jobs": -1} | random_state: 42


## 3. Vencedor do ciclo

O vencedor tem a melhor métrica primária na validação. Em caso de empate exato, vence o primeiro modelo na ordem de `config.models`.

In [5]:
winner = mdl.select_winner(results, cfg)
payload = mdl.save_competition(results, winner, ctx, cfg, ROOT)
print(f"Vencedor: {winner.label} ({winner.algorithm}) com {metric} = {winner.validation[metric]:.4f} na validação")
print("model_uri:", winner.model_uri)

Vencedor: modelo_B (random_forest) com f1 = 0.5343 na validação
model_uri: models:/m-5ce64fb2ca5448f88f77d6cb68bba4bc


## Conclusões

- Os dois algoritmos usaram os mesmos atributos, os mesmos dados de treino, a mesma validação, a mesma métrica e a mesma semente. As asserções acima interrompem o pipeline se isso deixar de valer.
- Hiperparâmetros, sementes, atributos, métricas e o modelo serializado estão no MLflow. Pela interface é possível comparar o modelo A com o B e um ciclo com outro, filtrando por `tags.cycle_id` ou `tags.model_name`.
- O vencedor ainda **não** está em produção. O notebook 04 decide se ele substitui o modelo vigente.

**Próximo passo:** `04_selecao_promocao.ipynb`.